# 📊 Task 1: Synthetic Data Generation for UAV Telemetry

---

## 🎯 Objective
Generate realistic synthetic telemetry data for UAVs operating in diverse environments (clear, windy, dusty terrains) to train our predictive models.

## 📚 Learning Objectives
1. Understand UAV battery physics and discharge characteristics
2. Model environmental factors affecting flight
3. Create feature-rich dataset for ML training

---

## 🔬 Mathematical Foundation: UAV Battery Physics

### 1. State of Charge (SoC) Model

The **State of Charge (SoC)** represents the remaining capacity of a battery as a percentage:

$$SoC(t) = SoC_0 - \int_0^t \frac{I(\tau)}{Q_{max}} d\tau$$

Where:
- $SoC_0$ = Initial state of charge (typically 100%)
- $I(\tau)$ = Current draw at time $\tau$ (Amperes)
- $Q_{max}$ = Maximum battery capacity (Ampere-hours)

**Simplified discrete version:**
$$SoC_{t+1} = SoC_t - \frac{I_t \cdot \Delta t}{Q_{max}} \times 100$$

---

### 2. Battery Voltage Model (Shepherd Model)

The terminal voltage of a Li-Po battery can be modeled as:

$$V(t) = V_{oc} - R_i \cdot I(t) - K \cdot \frac{Q_{max}}{Q_{max} - \int I d\tau} \cdot \int I d\tau$$

Simplified for our simulation:
$$V(SoC) = V_{min} + (V_{max} - V_{min}) \cdot SoC$$

Where:
- $V_{max}$ = Full charge voltage (~4.2V per cell)
- $V_{min}$ = Cutoff voltage (~3.0V per cell)
- For 6S battery: $V_{max} = 25.2V$, $V_{min} = 18V$

---

### 3. Power Consumption Model

UAV power consumption is affected by multiple factors:

$$P_{total} = P_{hover} + P_{forward} + P_{payload} + P_{env}$$

- **Hover Power**: $P_{hover} = \frac{(m_{uav} \cdot g)^{3/2}}{\sqrt{2 \rho A}}$
- **Forward Flight**: $P_{forward} = \frac{1}{2} \rho v^3 C_d A_f$
- **Payload Effect**: $P_{payload} \propto m_{payload}^{1.5}$
- **Environmental**: $P_{env} = P_{wind} + P_{dust}$

---

### 4. Range Estimation Model

Maximum flight range depends on available energy and power consumption:

$$Range = \frac{E_{available}}{P_{total}} \times v_{cruise}$$

Where:
$$E_{available} = SoC \times Q_{max} \times V_{nominal} \times \eta_{discharge}$$

---

### 5. Environmental Impact Factors

| Factor | Impact on Power | Mathematical Model |
|--------|-----------------|--------------------|
| Wind Speed | +15-40% | $\Delta P = k_w \cdot v_{wind}^2$ |
| Dust Level | +5-15% | $\Delta P = k_d \cdot dust_{level}$ |
| Temperature | Variable | $\eta(T) = 1 - 0.02 \cdot |T - T_{opt}|$ |
| Altitude | +5-20% | $\Delta P = k_a \cdot h^{0.5}$ |

---

## 🔄 Workflow Diagram

```
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Define UAV     │ ──▶ │  Generate Base  │ ──▶ │  Add Environment│
│  Parameters     │     │  Battery Data   │     │  Variables      │
└─────────────────┘     └─────────────────┘     └─────────────────┘
                                                        │
                                                        ▼
┌─────────────────┐     ┌─────────────────┐     ┌─────────────────┐
│  Save Dataset   │ ◀── │  Calculate      │ ◀── │  Compute Flight │
│  to CSV         │     │  Target Labels  │     │  Parameters     │
└─────────────────┘     └─────────────────┘     └─────────────────┘
```

## 📦 Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings

# Configuration
warnings.filterwarnings('ignore')
np.random.seed(42)  # For reproducibility
plt.style.use('seaborn-v0_8-whitegrid')

print("Libraries imported successfully!")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## ⚙️ Step 1: Define UAV System Parameters

We'll model a delivery/inspection UAV with the following specifications:

In [ ]:
# =============================================================================
# UAV System Parameters
# =============================================================================

class UAVParameters:
    """Configuration class for UAV specifications."""
    
    # Battery specifications (6S Li-Po)
    BATTERY_CELLS = 6
    VOLTAGE_PER_CELL_MAX = 4.2  # Volts (fully charged)
    VOLTAGE_PER_CELL_MIN = 3.0  # Volts (cutoff)
    BATTERY_CAPACITY = 10000    # mAh
    
    # Computed voltage ranges
    V_MAX = BATTERY_CELLS * VOLTAGE_PER_CELL_MAX  # 25.2V
    V_MIN = BATTERY_CELLS * VOLTAGE_PER_CELL_MIN  # 18.0V
    V_NOMINAL = BATTERY_CELLS * 3.7               # 22.2V
    
    # UAV physical specifications
    MASS_UAV = 3.5      # kg (without payload)
    MAX_PAYLOAD = 2.0   # kg
    
    # Flight characteristics
    CRUISE_SPEED = 15   # m/s (~54 km/h)
    MAX_ALTITUDE = 120  # meters
    
    # Power consumption (baseline)
    HOVER_POWER = 350   # Watts
    CRUISE_POWER = 450  # Watts
    
    # Environmental coefficients
    WIND_POWER_COEFF = 2.5     # Power increase per (m/s)^2 of wind
    DUST_POWER_COEFF = 0.5     # Power increase per dust unit
    TEMP_OPTIMAL = 25          # Optimal temperature (°C)
    TEMP_EFFICIENCY_DROP = 0.02  # Efficiency drop per °C deviation

# Display parameters
params = UAVParameters()
print("=" * 50)
print("          UAV SYSTEM PARAMETERS")
print("=" * 50)
print(f"Battery: {params.BATTERY_CELLS}S Li-Po, {params.BATTERY_CAPACITY}mAh")
print(f"Voltage Range: {params.V_MIN}V - {params.V_MAX}V")
print(f"UAV Mass: {params.MASS_UAV}kg (+ up to {params.MAX_PAYLOAD}kg payload)")
print(f"Cruise Speed: {params.CRUISE_SPEED}m/s")
print(f"Cruise Power: {params.CRUISE_POWER}W")
print("=" * 50)

## 🔋 Step 2: Battery Model Functions

Implementing the mathematical models defined earlier:

In [ ]:
def calculate_voltage(soc, v_max=UAVParameters.V_MAX, v_min=UAVParameters.V_MIN):
    """
    Calculate battery voltage based on State of Charge.
    
    Uses simplified linear model with realistic non-linearity at extremes.
    
    Parameters:
    -----------
    soc : float or ndarray
        State of Charge (0-100%)
    
    Returns:
    --------
    voltage : float or ndarray
        Terminal voltage in Volts
    """
    # Normalize SoC to 0-1 range
    soc_normalized = np.clip(soc / 100, 0, 1)
    
    # Non-linear voltage curve (typical Li-Po behavior)
    # Voltage drops faster at low SoC
    voltage_factor = 0.8 * soc_normalized + 0.2 * (soc_normalized ** 0.5)
    
    voltage = v_min + (v_max - v_min) * voltage_factor
    
    return voltage


def calculate_current(power, voltage):
    """
    Calculate current draw from power and voltage.
    
    I = P / V (Ohm's law)
    
    Parameters:
    -----------
    power : float or ndarray
        Power consumption in Watts
    voltage : float or ndarray
        Terminal voltage in Volts
    
    Returns:
    --------
    current : float or ndarray
        Current in Amperes
    """
    return power / voltage


def calculate_temperature(base_temp, current, altitude, ambient_temp=25):
    """
    Estimate battery temperature based on current draw and conditions.
    
    T_battery = T_ambient + delta_T_current - delta_T_altitude
    
    Parameters:
    -----------
    base_temp : float
        Base ambient temperature (°C)
    current : float
        Current draw (Amperes)
    altitude : float
        Flight altitude (meters)
    
    Returns:
    --------
    temperature : float
        Estimated battery temperature (°C)
    """
    # Temperature rise due to current (I²R heating)
    temp_rise_current = 0.5 * (current ** 1.5)
    
    # Temperature drop due to altitude (lapse rate ~6.5°C/1000m)
    temp_drop_altitude = 0.0065 * altitude
    
    temperature = base_temp + temp_rise_current - temp_drop_altitude
    
    # Realistic bounds
    return np.clip(temperature, -10, 65)


# Visualize the voltage-SoC relationship
soc_range = np.linspace(0, 100, 100)
voltage_range = calculate_voltage(soc_range)

plt.figure(figsize=(10, 5))
plt.subplot(1, 2, 1)
plt.plot(soc_range, voltage_range, 'b-', linewidth=2)
plt.xlabel('State of Charge (%)', fontsize=12)
plt.ylabel('Voltage (V)', fontsize=12)
plt.title('Battery Voltage vs SoC Curve', fontsize=14)
plt.grid(True, alpha=0.3)
plt.axhline(y=UAVParameters.V_MIN, color='r', linestyle='--', label=f'Cutoff ({UAVParameters.V_MIN}V)')
plt.axhline(y=UAVParameters.V_MAX, color='g', linestyle='--', label=f'Full ({UAVParameters.V_MAX}V)')
plt.legend()

plt.subplot(1, 2, 2)
current_range = np.linspace(5, 40, 100)
temp_range = [calculate_temperature(25, i, 50) for i in current_range]
plt.plot(current_range, temp_range, 'r-', linewidth=2)
plt.xlabel('Current (A)', fontsize=12)
plt.ylabel('Battery Temperature (°C)', fontsize=12)
plt.title('Temperature vs Current Draw', fontsize=14)
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/battery_characteristics.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Battery model functions created and visualized!")

## 🌍 Step 3: Environmental Factor Models

Model the impact of different terrains on UAV performance:

In [ ]:
# =============================================================================
# Environmental Models
# =============================================================================

TERRAIN_TYPES = {
    'clear': {
        'wind_mean': 3.0,      # m/s
        'wind_std': 1.5,
        'dust_mean': 10,       # µg/m³
        'dust_std': 5,
        'power_multiplier': 1.0
    },
    'windy': {
        'wind_mean': 12.0,     # m/s
        'wind_std': 4.0,
        'dust_mean': 25,       # µg/m³
        'dust_std': 10,
        'power_multiplier': 1.3
    },
    'dusty': {
        'wind_mean': 6.0,      # m/s
        'wind_std': 2.5,
        'dust_mean': 150,      # µg/m³
        'dust_std': 50,
        'power_multiplier': 1.15
    }
}


def calculate_power_consumption(base_power, wind_speed, dust_level, payload_weight, 
                                altitude, temperature):
    """
    Calculate total power consumption considering all factors.
    
    P_total = P_base × (1 + Σ adjustment_factors)
    
    Parameters:
    -----------
    base_power : float
        Base cruise power (Watts)
    wind_speed : float
        Wind speed (m/s)
    dust_level : float
        Dust concentration (µg/m³)
    payload_weight : float
        Payload weight (kg)
    altitude : float
        Flight altitude (m)
    temperature : float
        Ambient temperature (°C)
    
    Returns:
    --------
    power : float
        Total power consumption (Watts)
    """
    # Wind impact: quadratic relationship
    wind_factor = 1 + UAVParameters.WIND_POWER_COEFF * (wind_speed / 10) ** 2
    
    # Dust impact: linear relationship (motor efficiency loss)
    dust_factor = 1 + UAVParameters.DUST_POWER_COEFF * (dust_level / 100)
    
    # Payload impact: increases with weight^1.5 (physics of flight)
    payload_factor = 1 + 0.3 * (payload_weight / UAVParameters.MAX_PAYLOAD) ** 1.5
    
    # Altitude impact: air density decreases
    altitude_factor = 1 + 0.001 * altitude
    
    # Temperature efficiency (optimal at 25°C)
    temp_deviation = abs(temperature - UAVParameters.TEMP_OPTIMAL)
    temp_factor = 1 + UAVParameters.TEMP_EFFICIENCY_DROP * temp_deviation
    
    # Total power
    total_power = base_power * wind_factor * dust_factor * payload_factor * altitude_factor * temp_factor
    
    return total_power


def calculate_max_range(soc, power, cruise_speed=UAVParameters.CRUISE_SPEED):
    """
    Calculate maximum flight range based on available energy.
    
    Range = (Energy_available / Power) × Speed
    
    Parameters:
    -----------
    soc : float
        Current State of Charge (%)
    power : float
        Current power consumption (Watts)
    cruise_speed : float
        Cruise speed (m/s)
    
    Returns:
    --------
    range_km : float
        Maximum range in kilometers
    """
    # Available energy (Watt-hours)
    battery_capacity_wh = (UAVParameters.BATTERY_CAPACITY / 1000) * UAVParameters.V_NOMINAL
    available_energy = (soc / 100) * battery_capacity_wh * 0.9  # 90% usable
    
    # Flight time (hours)
    flight_time_hours = available_energy / power
    
    # Range (km) = speed (m/s) × time (h) × 3.6
    range_km = cruise_speed * flight_time_hours * 3.6
    
    return range_km


# Visualize power consumption factors
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Wind impact
wind_speeds = np.linspace(0, 20, 50)
wind_powers = [calculate_power_consumption(450, w, 10, 0.5, 50, 25) for w in wind_speeds]
axes[0, 0].plot(wind_speeds, wind_powers, 'b-', linewidth=2)
axes[0, 0].set_xlabel('Wind Speed (m/s)')
axes[0, 0].set_ylabel('Power (W)')
axes[0, 0].set_title('Power vs Wind Speed')
axes[0, 0].grid(True, alpha=0.3)

# Dust impact
dust_levels = np.linspace(0, 200, 50)
dust_powers = [calculate_power_consumption(450, 5, d, 0.5, 50, 25) for d in dust_levels]
axes[0, 1].plot(dust_levels, dust_powers, 'orange', linewidth=2)
axes[0, 1].set_xlabel('Dust Level (µg/m³)')
axes[0, 1].set_ylabel('Power (W)')
axes[0, 1].set_title('Power vs Dust Level')
axes[0, 1].grid(True, alpha=0.3)

# Payload impact
payloads = np.linspace(0, 2, 50)
payload_powers = [calculate_power_consumption(450, 5, 10, p, 50, 25) for p in payloads]
axes[1, 0].plot(payloads, payload_powers, 'g-', linewidth=2)
axes[1, 0].set_xlabel('Payload Weight (kg)')
axes[1, 0].set_ylabel('Power (W)')
axes[1, 0].set_title('Power vs Payload Weight')
axes[1, 0].grid(True, alpha=0.3)

# SoC vs Range for different conditions
soc_values = np.linspace(10, 100, 50)
for terrain, color in [('clear', 'green'), ('windy', 'red'), ('dusty', 'orange')]:
    powers = [calculate_power_consumption(450, TERRAIN_TYPES[terrain]['wind_mean'],
                                         TERRAIN_TYPES[terrain]['dust_mean'], 0.5, 50, 25)]
    ranges = [calculate_max_range(s, powers[0]) for s in soc_values]
    axes[1, 1].plot(soc_values, ranges, color=color, linewidth=2, label=terrain.capitalize())

axes[1, 1].set_xlabel('State of Charge (%)')
axes[1, 1].set_ylabel('Max Range (km)')
axes[1, 1].set_title('Range vs SoC for Different Terrains')
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../reports/figures/power_consumption_factors.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Environmental models created and visualized!")

## 📊 Step 4: Generate Synthetic Dataset

Now we'll generate the complete synthetic dataset with all features:

In [ ]:
def generate_uav_telemetry(n_samples=5000, random_state=42):
    """
    Generate synthetic UAV telemetry data.
    
    Parameters:
    -----------
    n_samples : int
        Number of samples to generate
    random_state : int
        Random seed for reproducibility
    
    Returns:
    --------
    df : pandas.DataFrame
        Generated telemetry dataset
    """
    np.random.seed(random_state)
    
    # Initialize lists to store data
    data = []
    
    # Generate timestamps (simulating real-time data over 30 days)
    start_time = datetime(2024, 1, 1, 0, 0, 0)
    
    for i in range(n_samples):
        # Randomly select terrain type
        terrain = np.random.choice(['clear', 'windy', 'dusty'], 
                                   p=[0.5, 0.3, 0.2])  # Clear more common
        terrain_config = TERRAIN_TYPES[terrain]
        
        # Generate timestamp
        timestamp = start_time + timedelta(minutes=i * 10 + np.random.randint(0, 5))
        
        # Generate environmental conditions
        wind_speed = max(0, np.random.normal(terrain_config['wind_mean'], 
                                              terrain_config['wind_std']))
        dust_level = max(0, np.random.normal(terrain_config['dust_mean'], 
                                              terrain_config['dust_std']))
        ambient_temp = np.random.normal(25, 10)  # Mean 25°C, std 10°C
        ambient_temp = np.clip(ambient_temp, -5, 45)  # Realistic range
        
        # Generate flight parameters
        altitude = np.random.uniform(10, UAVParameters.MAX_ALTITUDE)
        payload_weight = np.random.uniform(0, UAVParameters.MAX_PAYLOAD)
        flight_speed = np.random.uniform(8, 18)  # m/s
        
        # Generate battery state
        battery_soc = np.random.uniform(15, 100)  # 15-100% SoC
        
        # Calculate derived values
        voltage = calculate_voltage(battery_soc)
        
        # Calculate power consumption
        base_power = UAVParameters.CRUISE_POWER * (flight_speed / UAVParameters.CRUISE_SPEED)
        power_consumption = calculate_power_consumption(
            base_power, wind_speed, dust_level, payload_weight, altitude, ambient_temp
        )
        
        # Calculate current and battery temperature
        current = calculate_current(power_consumption, voltage)
        battery_temp = calculate_temperature(ambient_temp, current, altitude)
        
        # Calculate max range (target variable for regression)
        max_range = calculate_max_range(battery_soc, power_consumption, flight_speed)
        
        # Determine mission success (target for classification)
        # Mission is successful if sufficient range for planned distance
        planned_distance = np.random.uniform(1, max_range * 1.2)  # Some ambitious plans
        
        # Success factors: sufficient battery, not too windy, reasonable temp
        success_prob = (
            0.5 * (max_range / planned_distance if planned_distance > 0 else 1) +
            0.2 * (1 - wind_speed / 20) +
            0.15 * (1 - abs(battery_temp - 25) / 40) +
            0.15 * (battery_soc / 100)
        )
        success_prob = np.clip(success_prob, 0, 1)
        mission_success = 1 if np.random.random() < success_prob else 0
        
        # Create record
        record = {
            'timestamp': timestamp,
            'flight_id': f'FL_{i:05d}',
            
            # Battery features
            'battery_soc': round(battery_soc, 2),
            'voltage': round(voltage, 2),
            'current': round(current, 2),
            'battery_temperature': round(battery_temp, 2),
            
            # Environmental features
            'terrain_type': terrain,
            'wind_speed': round(wind_speed, 2),
            'dust_level': round(dust_level, 2),
            'ambient_temperature': round(ambient_temp, 2),
            
            # Flight parameters
            'altitude': round(altitude, 2),
            'payload_weight': round(payload_weight, 2),
            'flight_speed': round(flight_speed, 2),
            'planned_distance': round(planned_distance, 2),
            
            # Computed features
            'power_consumption': round(power_consumption, 2),
            
            # Target variables
            'max_range': round(max_range, 2),        # Regression target
            'mission_success': mission_success,      # Classification target
            'success_probability': round(success_prob, 4)
        }
        
        data.append(record)
    
    df = pd.DataFrame(data)
    return df


# Generate the dataset
print("Generating synthetic UAV telemetry data...")
df = generate_uav_telemetry(n_samples=5000)

print(f"\n✅ Dataset generated successfully!")
print(f"   Shape: {df.shape}")
print(f"   Columns: {len(df.columns)}")
print(f"   Memory: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

## 👀 Step 5: Explore the Generated Data

In [ ]:
# Display first few rows
print("\n" + "=" * 70)
print("                    DATASET PREVIEW")
print("=" * 70)
df.head(10)

In [ ]:
# Dataset info
print("\n" + "=" * 70)
print("                    DATASET INFO")
print("=" * 70)
df.info()

In [ ]:
# Statistical summary
print("\n" + "=" * 70)
print("                 STATISTICAL SUMMARY")
print("=" * 70)
df.describe().round(2)

## 📈 Step 6: Data Visualizations

In [ ]:
# Create comprehensive visualizations
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

# 1. SoC Distribution
axes[0, 0].hist(df['battery_soc'], bins=30, color='skyblue', edgecolor='black')
axes[0, 0].set_xlabel('Battery SoC (%)')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Battery SoC Distribution')

# 2. Max Range Distribution
axes[0, 1].hist(df['max_range'], bins=30, color='lightgreen', edgecolor='black')
axes[0, 1].set_xlabel('Max Range (km)')
axes[0, 1].set_ylabel('Frequency')
axes[0, 1].set_title('Maximum Range Distribution')

# 3. Terrain Type Distribution
terrain_counts = df['terrain_type'].value_counts()
axes[0, 2].pie(terrain_counts, labels=terrain_counts.index, autopct='%1.1f%%',
               colors=['lightgreen', 'lightcoral', 'khaki'])
axes[0, 2].set_title('Terrain Type Distribution')

# 4. SoC vs Max Range (colored by terrain)
for terrain, color in [('clear', 'green'), ('windy', 'red'), ('dusty', 'orange')]:
    mask = df['terrain_type'] == terrain
    axes[1, 0].scatter(df.loc[mask, 'battery_soc'], df.loc[mask, 'max_range'],
                       c=color, alpha=0.5, label=terrain, s=10)
axes[1, 0].set_xlabel('Battery SoC (%)')
axes[1, 0].set_ylabel('Max Range (km)')
axes[1, 0].set_title('SoC vs Max Range by Terrain')
axes[1, 0].legend()

# 5. Power Consumption Distribution
axes[1, 1].hist(df['power_consumption'], bins=30, color='coral', edgecolor='black')
axes[1, 1].set_xlabel('Power Consumption (W)')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Power Consumption Distribution')

# 6. Mission Success by Terrain
success_by_terrain = df.groupby('terrain_type')['mission_success'].mean()
axes[1, 2].bar(success_by_terrain.index, success_by_terrain.values,
               color=['lightgreen', 'khaki', 'lightcoral'])
axes[1, 2].set_xlabel('Terrain Type')
axes[1, 2].set_ylabel('Success Rate')
axes[1, 2].set_title('Mission Success Rate by Terrain')
axes[1, 2].set_ylim(0, 1)

# 7. Wind Speed vs Power Consumption
axes[2, 0].scatter(df['wind_speed'], df['power_consumption'], alpha=0.3, s=5, c='purple')
axes[2, 0].set_xlabel('Wind Speed (m/s)')
axes[2, 0].set_ylabel('Power Consumption (W)')
axes[2, 0].set_title('Wind Speed vs Power')

# 8. Payload vs Range
axes[2, 1].scatter(df['payload_weight'], df['max_range'], alpha=0.3, s=5, c='teal')
axes[2, 1].set_xlabel('Payload Weight (kg)')
axes[2, 1].set_ylabel('Max Range (km)')
axes[2, 1].set_title('Payload vs Range')

# 9. Current vs Battery Temperature
axes[2, 2].scatter(df['current'], df['battery_temperature'], alpha=0.3, s=5, c='crimson')
axes[2, 2].set_xlabel('Current (A)')
axes[2, 2].set_ylabel('Battery Temperature (°C)')
axes[2, 2].set_title('Current vs Battery Temperature')

plt.tight_layout()
plt.savefig('../reports/figures/data_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✅ Visualizations created and saved!")

## 💾 Step 7: Save the Dataset

In [ ]:
# Save to CSV
output_path = '../data/raw/uav_telemetry.csv'
df.to_csv(output_path, index=False)

print(f"\n✅ Dataset saved to: {output_path}")
print(f"   Total records: {len(df)}")
print(f"   File size: {df.memory_usage(deep=True).sum() / 1024:.2f} KB")

# Also save a summary
summary = {
    'total_samples': len(df),
    'features': list(df.columns),
    'terrain_distribution': df['terrain_type'].value_counts().to_dict(),
    'success_rate': df['mission_success'].mean(),
    'soc_range': [df['battery_soc'].min(), df['battery_soc'].max()],
    'range_stats': {
        'mean': df['max_range'].mean(),
        'std': df['max_range'].std(),
        'min': df['max_range'].min(),
        'max': df['max_range'].max()
    }
}

print("\n" + "=" * 50)
print("           DATASET SUMMARY")
print("=" * 50)
print(f"Total Samples: {summary['total_samples']}")
print(f"Overall Success Rate: {summary['success_rate']:.2%}")
print(f"\nTerrain Distribution:")
for terrain, count in summary['terrain_distribution'].items():
    print(f"  - {terrain}: {count} ({count/len(df):.1%})")
print(f"\nRange Statistics:")
print(f"  - Mean: {summary['range_stats']['mean']:.2f} km")
print(f"  - Std: {summary['range_stats']['std']:.2f} km")
print(f"  - Min: {summary['range_stats']['min']:.2f} km")
print(f"  - Max: {summary['range_stats']['max']:.2f} km")

---

## 📚 Learning Resources

### Blogs & Articles
1. **Battery Modeling**: [Understanding Li-Po Battery Characteristics](https://batteryuniversity.com/article/bu-409-charging-lithium-ion)
2. **UAV Power Consumption**: [Multirotor Power Theory](https://www.rcgroups.com/forums/showthread.php?1968070-Power-consumption-of-multirotors)
3. **Synthetic Data Generation**: [Synthetic Data for ML](https://towardsdatascience.com/synthetic-data-generation-a-must-have-skill-for-new-data-scientists-915896c0c1ae)

### YouTube Videos
1. **NumPy Fundamentals**: [NumPy Tutorial for Beginners](https://www.youtube.com/watch?v=QUT1VHiLmmI)
2. **Pandas Data Analysis**: [Pandas Complete Tutorial](https://www.youtube.com/watch?v=vmEHCJofslg)
3. **UAV Battery Systems**: [Drone Battery Explained](https://www.youtube.com/watch?v=ofrYr_q5xyw)

### Research Papers
1. Traub, L.W. (2011). "Range and Endurance Estimates for Battery-Powered Aircraft"
2. Gatti, M. et al. (2015). "Maximum Endurance for Battery-Powered Rotary-Wing Aircraft"

---

## ✅ Task 1 Complete!

### What We Accomplished:
- ✅ Understood UAV battery physics (SoC, voltage, current relationships)
- ✅ Modeled environmental impact factors (wind, dust, temperature)
- ✅ Created power consumption model
- ✅ Generated 5000 synthetic telemetry samples
- ✅ Created comprehensive visualizations
- ✅ Saved dataset for next tasks

### Next Task: Data Preprocessing Pipeline
We'll clean, normalize, and prepare the data for machine learning.

---